# FLAG 2027 - Notebook 24: voice DIVERSITY (paralinguistic + multilingual blocks) and a language-nuisance projection

Why: the paralinguistic audeering model gave a bigger gain (-0.7) than another VoxCeleb speaker encoder (-0.5); stacking more of the same family saturates. Three frozen models from different families, all public:

| Name | Model | What it adds |
|---|---|---|
| `e2v` | emotion2vec+ large (FunASR `iic/emotion2vec_plus_large`, data2vec-style SSL, multilingual emotion fine-tune) | emotion / speaking style |
| `whisper` | openai/whisper-large-v3 ENCODER, mean of layers 20-28, pooled over the clip's real frames (1280-d) | multilingual acoustics incl. Bengali (frozen, allowed, declared); speaker info is in the deep layers |
| `campp` | 3D-Speaker CAM++ zh_en advanced (ONNX, 192-d), supervised on a Chinese+English corpus, NOT VoxCeleb | a different speaker space |

Plus **NAP** (nuisance-attribute projection): MAV speakers were recorded in two languages each (English+Urdu, English+Hindi, English+German). The per-speaker difference between their two language means is a pure 'language' direction. We take the top-k such directions (SVD) from TRAINING data only and project them out of the voice features. Fixed before test, no test data involved: allowed.

Steps: voice file lists in our row order (alignment check) -> extract the three encoders -> CV: each added to R4, NAP on R4, then the best combination -> ZIP (no VoxCeleb). Reference S0 = 25.53 reused from NB21. About 3-4 h. Resumable. Paste the **SUMMARY**.

In [ ]:
# Cell 1 - Drive, GPU, packages
from google.colab import drive
drive.mount('/content/drive')
!pip -q install speechbrain onnxruntime funasr modelscope
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('[v1 28-Oct] Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - voice file lists in our row order (+ language of every MAV row, for NAP) + alignment check with the official ECAPA
import os, re, sys, json, time, zipfile, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.endswith(('.wav', '.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def wav_token(tokens):
    t = [x for x in tokens if x.lower().endswith('.wav')]; assert len(t) == 1, tokens; return t[0]
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(wav_token(l.split()), lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(wav_token(l.split()), lp) for l in open(lp) if l.strip()]
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_paths, mspk, mlang = [], [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, vm)); mspk.append(ver + '_' + t[0].split('/')[0]); mlang.append(lf.split('_')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            _ = fr[rng.integers(len(fr))]
            if need:
                zf.extract(vn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn), lang_))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]; xlang = [r[2] for r, k in zip(rows, keep) if k]
EXT_LANG = np.array(mlang + xlang); EXT_SPK = np.array(list(MAV['spk']) + list(EXTRA['spk']))
try:
    from speechbrain.inference.interfaces import Pretrained
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
rng = np.random.default_rng(1)
for k, saved in {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}.items():
    idx = rng.choice(len(saved), 100, replace=False)
    new = np.stack([ECAPA.encode_batch(torch.from_numpy(librosa.load(FILES[k][i], sr=16000, mono=True)[0]).unsqueeze(0)).squeeze().cpu().numpy() for i in idx]); old = saved[idx]
    cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    note('alignment %-6s median cosine %.4f' % (k, np.median(cos))); assert np.median(cos) > 0.98
del ECAPA; torch.cuda.empty_cache()
note('Files: ' + ', '.join('%s %d' % (k, len(v)) for k, v in FILES.items()) + ' | MAV languages: ' + ', '.join('%s %d' % (l, (EXT_LANG == l).sum()) for l in sorted(set(EXT_LANG))))

In [ ]:
# Cell 3 - the three encoders
def crop(w, sec):
    L = 16000 * sec
    return w[(len(w) - L) // 2:(len(w) - L) // 2 + L] if len(w) >= L else np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]

def enc_e2v(paths):
    from funasr import AutoModel
    m = AutoModel(model='iic/emotion2vec_plus_large', hub='ms', disable_update=True, device=dev); out = []
    for p in tqdm(paths, desc='emotion2vec+'):
        w = librosa.load(p, sr=16000, mono=True)[0][:16000 * 20]
        r = m.generate(input=w, fs=16000, granularity='utterance', extract_embedding=True, disable_pbar=True)
        out.append(np.asarray(r[0]['feats'], np.float32).reshape(-1))
    del m; torch.cuda.empty_cache(); return np.stack(out)

def enc_whisper(paths, bs=8, layers=(20, 29)):
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    fe = WhisperFeatureExtractor.from_pretrained(name); enc = WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval(); out = []
    for i in tqdm(range(0, len(paths), bs), desc='whisper-v3 enc'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths[i:i + bs]]
        x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
        with torch.no_grad():
            hs = enc(x, output_hidden_states=True).hidden_states
        h = torch.stack(hs[layers[0]:layers[1]]).float().mean(0)                 # [B, 1500, 1280]
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); out.append(h[k_, :n].mean(0).cpu().numpy())
    del enc; torch.cuda.empty_cache(); return np.stack(out)

def enc_campp(paths, sec=6):
    import onnxruntime as ort, torchaudio
    from huggingface_hub import hf_hub_download
    f = hf_hub_download('csukuangfj/speaker-embedding-models', '3dspeaker_speech_campplus_sv_zh_en_16k-common_advanced.onnx', local_dir=os.path.join(CACHE, 'campp'))
    sess = ort.InferenceSession(f, providers=['CPUExecutionProvider']); iname = sess.get_inputs()[0].name; out = []
    for p in tqdm(paths, desc='cam++ zh_en'):
        w = crop(librosa.load(p, sr=16000, mono=True)[0], sec)
        fb = torchaudio.compliance.kaldi.fbank(torch.from_numpy(w).unsqueeze(0) * (1 << 15), num_mel_bins=80, frame_length=25, frame_shift=10, dither=0.0, sample_frequency=16000, window_type='hamming', use_energy=False)
        fb = (fb - fb.mean(0, keepdim=True)).numpy()[None].astype(np.float32)
        out.append(sess.run(None, {iname: fb})[0].reshape(-1))
    return np.stack(out)
ENCODERS = {'campp': enc_campp, 'e2v': enc_e2v, 'whisper': enc_whisper}
ORDER = SPLIT_NAMES + ['mav', 'extra']
allp = [p for s in ORDER for p in FILES[s]]
def split_back(emb):
    out, i = {}, 0
    for s in ORDER:
        n = len(FILES[s]); out[s] = emb[i:i + n].astype(np.float32); i += n
    return out
def _l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def sanity(name, E):
    """on the TRAIN rows: same-identity cosine must exceed different-identity cosine (weak check for paralinguistic models; strong for speaker models)"""
    rng = np.random.default_rng(7); idx = rng.choice(len(FILES['train']), 300, replace=False); Z = _l2(E[idx]); lab = y[idx]; C = Z @ Z.T
    same = (lab[:, None] == lab[None, :]) & ~np.eye(len(idx), dtype=bool); diff = lab[:, None] != lab[None, :]
    note('%s sanity: same-id cosine %.3f | different-id %.3f | gap %.3f' % (name, C[same].mean(), C[diff].mean(), C[same].mean() - C[diff].mean()))
for name, fn in ENCODERS.items():
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        print(name, 'already extracted'); continue
    t0 = time.time()
    try:
        emb = fn(allp); d = split_back(emb); sanity(name, d['train']); np.savez(p, **d)
        note('%-7s dim %4d | %d files in %.1f min' % (name, emb.shape[1], len(allp), (time.time() - t0) / 60))
    except Exception as e:
        note('ENCODER %s FAILED: %r' % (name, repr(e)[:300])); torch.cuda.empty_cache()

In [ ]:
# Cell 4 - CV (challenge + all MAV, CLIP face, prototypes; 5-fold x 3 seeds): R4 + each block, NAP on R4, then the best combination
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in list(ENCODERS) + ['sam100', 'w2vb']:          # NB21 encoders are picked up too if they exist
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in ORDER}
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
def nap_basis(V_ext, k):
    """language-nuisance directions from TRAINING MAV rows: per speaker, mean(language A) - mean(language B); top-k right singular vectors"""
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return X - (X @ U.T) @ U
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
NAP_K = 8
def voice_matrix(vn_, nap):
    V = build_train(vn_)
    if nap:
        U, n_d = nap_basis(V[N_CH:], NAP_K); V = nap_apply(V, U); return V, U
    return V, None
CSV = os.path.join(RESDIR, 'nb24_cv.csv')
if not os.path.exists(CSV) and os.path.exists(os.path.join(RESDIR, 'nb21_cv.csv')):
    r21 = pd.read_csv(os.path.join(RESDIR, 'nb21_cv.csv')); r21 = r21[r21.combo == 'S0_R4_ref'].copy(); r21['combo'] = 'D0_R4'; r21.to_csv(CSV, index=False)
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
def run_combos(COMBOS):
    global done
    for name, (vn_, nap) in COMBOS.items():
        F_raw = build_train(['clip']); V_raw, _ = voice_matrix(vn_, nap)
        for seed in SEEDS:
            for k in range(K):
                if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                    continue
                tr_idx, vsets = FOLD_DATA[k]; model, best_ep, XF, XV, _ = train_combo(F_raw, V_raw, tr_idx, vsets, seed=seed)
                e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
                print('%-16s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
def table():
    R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
    return R, R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean().round(2)
STAGE1 = {'D0_R4': (R4, False), 'D1_R4_nap%d' % NAP_K: (R4, True)}
STAGE1.update({'D2_+' + n: (R4 + [n], False) for n in ENCODERS if n in FEAT})
run_combos(STAGE1); R, tab = table()
ref = tab.loc['D0_R4', 'overall']
wins = [c[4:] for c in tab.index if c.startswith('D2_+') and tab.loc[c, 'overall'] < ref - 0.15]
STAGE2 = {}
if len(wins) > 1:
    STAGE2['D3_+' + '+'.join(wins)] = (R4 + wins, False)
if wins and tab.loc['D1_R4_nap%d' % NAP_K, 'overall'] < ref - 0.15:
    STAGE2['D4_+' + '+'.join(wins) + '_nap'] = (R4 + wins, True)
run_combos(STAGE2); R, tab = table()
COMBOS = {**STAGE1, **STAGE2}
note('VOICE DIVERSITY, face = CLIP (challenge + all MAV, prototypes; 5-fold x 3 seeds)'); note(tab.to_string())
BEST = tab['overall'].idxmin(); note('BEST: %s (%.2f) vs R4 reference %.2f | blocks that helped alone: %s' % (BEST, tab.loc[BEST, 'overall'], ref, wins or 'none'))

In [ ]:
# Cell 4b [added 29-Oct] - UNHEARD-LANGUAGE proxy CV (CV-U). The English-only CV above cannot see the Bengali cells.
# Hold out bilingual MAV speakers (English + Urdu/Hindi/German); train on the challenge ids + the other MAV speakers;
# test held-out faces vs their NON-English voices ('unheard' proxy) and vs their English voices ('heard' proxy). 3 folds x 2 seeds, fixed epochs.
BIL = sorted(s for s in np.unique(EXT_SPK) if len(np.unique(EXT_LANG[EXT_SPK == s])) >= 2)
rng_u = np.random.default_rng(7); rng_u.shuffle(BIL); KU = 3; ufolds = [BIL[i::KU] for i in range(KU)]
note('CV-U: %d bilingual MAV speakers, %d folds of ~%d held-out speakers' % (len(BIL), KU, len(ufolds[0])))
def upairs(rows_v, spk_v, rows_f, spk_f, n=2000, seed=0):
    rng = np.random.default_rng(seed); spk_ = [s for s in np.unique(spk_v) if (spk_f == s).any()]
    rv = {s: rows_v[spk_v == s] for s in spk_}; rf = {s: rows_f[spk_f == s] for s in spk_}; V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rv[s])); F.append(rng.choice(rf[s])); L.append(1)
        s, t = rng.choice(spk_, 2, replace=False); V.append(rng.choice(rv[s])); F.append(rng.choice(rf[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
UFOLD = []
for k in range(KU):
    held = set(ufolds[k]); hm = np.array([s in held for s in EXT_SPK])
    rows_h, spk_h, lang_h = EXT[hm], EXT_SPK[hm], EXT_LANG[hm]
    pu = upairs(rows_h[lang_h != 'English'], spk_h[lang_h != 'English'], rows_h, spk_h, seed=k)
    ph = upairs(rows_h[lang_h == 'English'], spk_h[lang_h == 'English'], rows_h, spk_h, seed=50 + k)
    UFOLD.append((hm, [pu, ph]))
def nap_basis_mask(V_ext, k, mask):
    S_, L_, Vm = EXT_SPK[mask], EXT_LANG[mask], V_ext[mask]; diffs = []
    for s in np.unique(S_):
        m = S_ == s; langs = np.unique(L_[m])
        if len(langs) < 2:
            continue
        mu = [Vm[m & (L_ == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); return np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k].astype(np.float32)
CSVU = os.path.join(RESDIR, 'nb24_cvu.csv'); doneu = pd.read_csv(CSVU) if os.path.exists(CSVU) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
EXT_ALL = EXT.copy()
for name, (vn_, nap) in COMBOS.items():
    ep_fixed = int(np.median(R[R.combo == name]['best_ep'])); F_raw = build_train(['clip']); V_full = build_train(vn_)
    for k in range(KU):
        hm, vsets = UFOLD[k]
        V_raw = V_full
        if nap:
            U = nap_basis_mask(V_full[N_CH:], NAP_K, ~hm); V_raw = nap_apply(V_full, U)        # basis from TRAINING speakers only
        for seed in [1, 2]:
            if ((doneu.combo == name) & (doneu.seed == seed) & (doneu.fold == k)).any():
                continue
            EXT = EXT_ALL[~hm]                                                               # train_combo reads the global EXT: held-out speakers excluded
            try:
                model, _, XF, XV, _ = train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=seed, max_epochs=ep_fixed)
            finally:
                EXT = EXT_ALL
            e_u, e_h = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, unheard=e_u, heard=e_h)])
            row.to_csv(CSVU, mode='a', header=not os.path.exists(CSVU), index=False); doneu = pd.concat([doneu, row])
            print('CV-U %-16s seed %d fold %d | unheard %.2f | heard %.2f | %.0f min' % (name, seed, k, e_u, e_h, (time.time() - t0) / 60))
RU = pd.read_csv(CSVU); tabu = RU.groupby('combo')[['unheard', 'heard']].mean().round(2); tabu['gap'] = (tabu['unheard'] - tabu['heard']).round(2)
note('CV-U (held-out bilingual MAV speakers; faces vs NON-English voices = unheard proxy, vs English = heard proxy; 3 folds x 2 seeds)'); note(tabu.to_string())
note('CV-U best on UNHEARD: %s (%.2f) vs D0_R4 %.2f' % (tabu['unheard'].idxmin(), tabu['unheard'].min(), tabu.loc['D0_R4', 'unheard']))

In [ ]:
# Cell 5 - ZIP for the best combo (no VoxCeleb) + saved dev scores
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
vn_, nap = COMBOS[BEST]; F_raw = build_train(['clip']); V_raw, U = voice_matrix(vn_, nap)
ep = int(np.median(R[R.combo == BEST]['best_ep']))
finals = [train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=s, max_epochs=ep)[0] for s in SEEDS]
_, _, ch = dnorm(F_raw, V_raw, np.concatenate([np.arange(N_CH), EXT]))
sc = {}
for track in ['no_gender', 'gender']:
    for lang_ in LANG_FILE:
        split = track + '_' + lang_; vx = build(vn_, split)
        if U is not None:
            vx = nap_apply(vx, U)
        xf = torch.tensor(((build(['clip'], split) - ch[0]) / ch[1]).astype(np.float32), device=dev); xv = torch.tensor(((vx - ch[2]) / ch[3]).astype(np.float32), device=dev)
        sc[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
tag = BEST.replace('+', '_'); OUT = '/content/sub_24_' + tag
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang_, fname in LANG_FILE.items():
        split = track + '_' + lang_; pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
        assert len(pair_ids) == len(sc[split]) and np.isfinite(sc[split]).all()
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, -sc[split]):
                f.write('%s %.6f\n' % (pid, v))
zp = os.path.join(SUBDIR, '24_%s_noVox.zip' % tag)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_24_%s.npz' % tag), **sc); note('Submission ZIP saved: ' + zp[len(BASE):])
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)